# NOTEARS Quick Start Guide

Learn how to use NOTEARS for directed acyclic graph (DAG) structure learning from observational data.

**By the end of this notebook, you'll:**
- Install and verify NOTEARS
- Generate synthetic data from a known DAG
- Learn the DAG structure using NOTEARS
- Evaluate the learned structure
- Understand key hyperparameters

## Installation

Add NOTEARS to your `Cargo.toml`:

```toml
[dependencies]
notears = "0.1"
ndarray = "0.15"
```

## Example: Learn DAG Structure from Synthetic Data

Let's solve a simple problem step-by-step.

### Step 1: Create Synthetic Data

Generate data from a known DAG structure with 10 nodes and 1000 samples.

```rust
use notears::optimization::solve;
use notears::utils::standardize_data;
use notears::scoring::{mse_loss, l1_penalty};
use ndarray::Array2;
use rand::Rng;

fn main() -> Result<(), Box<dyn std::error::Error>> {
    // 1. Create synthetic data
    let mut rng = rand::thread_rng();
    let (n, d) = (1000, 10);
    
    // Generate random lower-triangular W (acyclic by construction)
    let mut w_true = Array2::zeros((d, d));
    for i in 0..d {
        for j in 0..i {
            if rng.gen::<f64>() < 0.4 {  // 40% edge density
                w_true[[i, j]] = rng.gen::<f64>() * 2.0 - 1.0;  // weight in [-1, 1]
            }
        }
    }
    
    // Generate data from structural equation model
    let mut data = Array2::zeros((n, d));
    for i in 0..n {
        // Exogenous noise
        for j in 0..d {
            data[[i, j]] = rng.gen::<f64>() * 2.0 - 1.0;
        }
        
        // Apply linear transformation
        for node in 0..d {
            for parent in 0..node {
                data[[i, node]] += w_true[[node, parent]] * data[[i, parent]];
            }
        }
    }
    
    println!("Generated {} samples with {} variables", n, d);
    println!("True DAG has {} edges", 
             w_true.iter().filter(|&&x| x.abs() > 0.01).count());
```

### Step 2: Standardize Data

Standardization (zero mean, unit variance) improves numerical stability.

```rust
    // 2. Standardize data
    let standardized = standardize_data(&data)?;
    
    // Verify standardization
    let (n, d) = standardized.dim();
    for j in 0..d {
        let col = standardized.column(j).to_owned();
        let mean: f64 = col.iter().sum::<f64>() / col.len() as f64;
        let var: f64 = col.iter().map(|x| (x - mean).powi(2)).sum::<f64>() / col.len() as f64;
        println!("Variable {}: mean={:.6}, std={:.6}", j, mean, var.sqrt());
    }
```

### Step 3: Learn DAG Structure

Solve NOTEARS with default configuration and λ=0.1.

```rust
    // 3. Learn DAG structure with lambda=0.1
    println!("\nSolving NOTEARS...");
    let result = solve(&standardized, 0.1)?;
    
    println!("Convergence status:");
    println!("  Iterations: {}", result.iterations);
    println!("  Constraint violation h(W): {:.2e}", result.constraint_violation);
    println!("  Final score F(W): {:.6e}", result.final_score);
    println!("  Acyclic: {}", result.is_acyclic());
    
    // Extract edges
    let learned_edges = result.edges();
    println!("\nLearned {} edges (threshold=0.3)", learned_edges.len());
```

### Step 4: Evaluate Results

Compare learned structure to ground truth and compute accuracy metrics.

```rust
    // 4. Evaluate
    let true_edges = w_true.iter()
        .enumerate()
        .filter_map(|(idx, &val)| {
            if val.abs() > 0.01 {
                let i = idx / d;
                let j = idx % d;
                Some((i, j))
            } else {
                None
            }
        })
        .collect::<Vec<_>>();
    
    let learned_edges_set = learned_edges.iter()
        .map(|(i, j, _)| (i, j))
        .collect::<Vec<_>>();
    
    // Compute metrics
    let tp = learned_edges_set.iter()
        .filter(|e| true_edges.contains(&(***e.0, ***e.1)))
        .count();
    
    let fp = learned_edges_set.len() - tp;
    let fn_count = true_edges.len() - tp;
    
    let precision = tp as f64 / (tp + fp).max(1) as f64;
    let recall = tp as f64 / (tp + fn_count).max(1) as f64;
    let f1 = 2.0 * (precision * recall) / (precision + recall).max(0.001);
    
    println!("\nEvaluation metrics:");
    println!("  True positives: {}", tp);
    println!("  False positives: {}", fp);
    println!("  False negatives: {}", fn_count);
    println!("  Precision: {:.3}", precision);
    println!("  Recall: {:.3}", recall);
    println!("  F1-score: {:.3}", f1);
    
    Ok(())
}
```

## Key Hyperparameters

### Lambda (λ): L1 Regularization Strength
- Controls sparsity of learned DAG
- Larger λ → fewer edges
- Try values: [0.01, 0.05, 0.1, 0.2, 0.5]

### Constraint Tolerance
- Convergence threshold for acyclicity constraint h(W)
- Default: 1e-8 (strict)
- Typical range: [1e-9, 1e-6]

### Edge Threshold
- Values below threshold treated as zero edges
- Default: 0.3
- Use 0.5-1.0 for very sparse solutions

## Next Steps

1. **Try different data regimes:**
   - Small n, large d (underdetermined): see docs/CONFIGURATION.md
   - Large n, small d (overdetermined): use stricter tolerance

2. **Parameter tuning:**
   - Grid search over λ values
   - Use cross-validation for optimal λ

3. **Production deployment:**
   - See README.md for integration examples
   - Review TROUBLESHOOTING.md for common issues

4. **Performance analysis:**
   - Check BENCHMARKING.md for profiling
   - Run `cargo bench` for performance metrics

## References

- [README.md](../README.md) - Installation and quick start
- [API Reference](../docs/API.md) - Function documentation
- [Configuration Guide](../docs/CONFIGURATION.md) - Tuning by data regime
- [Troubleshooting](../TROUBLESHOOTING.md) - Common issues and solutions